In [50]:
from pymongo import MongoClient
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["jotuns_lair"]

rooms    = db["rooms"]
users    = db["users"]
loot     = db["loot"]
monsters = db["monsters"]

### A. Consulta para extraer los comentarios y exportación a JSON

Actualmente los comentarios están almacenados como arrays embebidos en las colecciones `Rooms` y `Users`. Para centralizarlos en una única colección, necesitamos desanidar esos arrays y formatear los campos según la nueva estructura de la colección `Hints`.

Dado que un comentario pertenece inherentemente a una sala y dentro de él ya se incluye la información del usuario (subdocumento `publish_by`), la forma más sencilla y sin duplicados es **extraer todos los comentarios únicamente desde la colección `Rooms`**, que contiene la relación completa habitación‑usuario. Esto evita posibles discrepancias entre ambas colecciones.

**Pipeline de agregación en MongoDB** (para ejecutar en `mongosh` o en la sección *Aggregation* de Compass):

```javascript
mongodb.rooms.aggregate([
  {
    $unwind: "$hints"   // Separa cada comentario del array
  },
  {
    $project: {
      _id: 0,            // Se genera un nuevo _id automáticamente al exportar
      Creation_date: "$hints.creation_date",
      HintText: "$hints.text",
      Category: "$hints.category",
      References_room: {
        IdR: "$room_id",
        Name: "$room_name",
        IdD: "$dungeon_id",
        Dungeon: "$dungeon_name"
      },
      Publish_by: {
        Email: "$hints.publish_by.email",
        User_name: "$hints.publish_by.user_name",
        CreationDate: "$hints.publish_by.creation_date",
        Country: "$hints.publish_by.country"
      }
    }
  }
]);
```

**Exportación del resultado a JSON:**

- **Opción 1 (MongoDB Compass):** tras ejecutar la agregación, hacer clic en *Export*, *Export Full Collection* (o *Export Query Result*) y elegir formato JSON.
- **Opción 2 (script personalizado):** escribir un script en Python (usando `pymongo`) que devuelva los resultados y los escriba en un archivo JSON.

El archivo `hints.json` resultante contendrá los documentos con la estructura deseada.


### B. Creación de la colección `Hints`, importación y limpieza de las colecciones originales

1. **Crear la colección `Hints` e importar el JSON**  
   Desde la terminal:
   ```bash
   mongoimport --db mongodb --collection Hints --file hints.json --jsonArray
   ```
   Esto inserta todos los comentarios extraídos en la nueva colección. Se presupone que el nombre del archivo es `hints.json` y que el formato es un array de documentos JSON y que la base de datos se llama `mongodb`.

   También se puede usar MongoDB Compass para importar el archivo JSON directamente a la colección `Hints` (creando una nueva colección e importando el JSON usando Add Data).

2. **Eliminar el campo `hints` de las colecciones `Rooms` y `Users`**  
   Una vez verificada la correcta importación, se eliminan los arrays embebidos:

   ```javascript
   mongodb.rooms.updateMany({}, { $unset: { hints: "" } });
   mongodb.users.updateMany({}, { $unset: { hints: "" } });
   ```

   > **Nota 1:** También podríamos eliminar el campo `hints` de los usuarios si existiera, ya que ahora los comentarios residen en `Hints` y se referencian por email del usuario.
   
   > **Nota 2:** Podríamos también haber usado $out y $merge para crear la colección `Hints` directamente.




In [51]:
def eliminar_hints():

    rooms.update_many({}, {"$unset": {"hints": ""}})
    users.update_many({}, {"$unset": {"hints": ""}})
    

In [52]:
eliminar_hints()

### C. Adaptación de los endpoints al nuevo esquema

Ahora que los comentarios están en una colección independiente, las operaciones de lectura y escritura deben dirigirse a `Hints`. Se describen los cambios funcionales para cada endpoint.


#### `POST /comment`

Este endpoint añade un nuevo comentario. Recibe como parámetros: user_email (str), room_id (int), text (str), category (str).

*Antes:* Insertaba el comentario en el array `hints` de la sala y  también en el usuario.

*Ahora:*
- Inserta un único documento en `Hints` con los datos recibidos (`user_email`, `room_id`, `text`, `category`) y añade:
  - La fecha actual como `Creation_date`.
  - El subdocumento `References_room` rellenado consultando la sala (id, nombre, id de mazmorra, nombre de mazmorra).
  - El subdocumento `Publish_by` obtenido de la colección `Users` a partir del `user_email` (nombre, país, fecha de creación del usuario).
- Si se aplicó el **patrón Computed** (ejercicio anterior), se actualiza además los contadores de categorías en la sala correspondiente.


In [53]:
hints = db["hints"]

In [54]:
def post_comment(user_email, room_id, text, category):
    user = users.find_one({"email": user_email})
    room = rooms.find_one({"room_id": room_id})
    new_comment = {
        "Creation_date": datetime.utcnow().isoformat(),
        "HintText": text,
        "Category": category,
        "References_room": {
            "IdR": room["room_id"],
            "Name": room["room_name"],
            "IdD": room["dungeon_id"],
            "Dungeon": room["dungeon_name"]
        },
        "Publish_by": {
            "Email": user_email,
            "User_name": user["user_name"],
            "CreationDate": user["creation_date"],
            "Country": user["country"]
        }
    }
    hints.insert_one(new_comment)

    """
    Como no hemos realizado los patrones de diseño, lo dejamos aquí únicamente como comentario. 
    db.Rooms.update_one(
        {"room_id": room_id},
        {"$inc": {f"comments_by_category.{category}": 1}}
    )
    """

In [55]:
post_comment("adanherranz@example.org", 2, "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.", "General")

Comprobamos que funciona correctamente. 

In [ ]:
def get_comment(email, room_id, category=None):
    query = {
        "Publish_by.Email": email,
        "References_room.IdR": room_id
    }
    if category:
        query["Category"] = category
    return list(hints.find(query))

In [57]:
get_comment("adanherranz@example.org", 2, "General")

[{'_id': ObjectId('6a0439802373cf21fcd63687'),
  'Creation_date': '2026-05-13T08:42:40.445679',
  'HintText': "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.",
  'Category': 'General',
  'References_room': {'IdR': 2,
   'Name': 'sanctuary ',
   'IdD': 0,
   'Dungeon': 'Burghap, Prison of the Jealous Hippies'},
  'Publish_by': {'Email': 'adanherranz@example.org',
   'User_name': 'auroraespana',
   'CreationDate': '2022-01-02',
   'Country': 'es_ES'}},
 {'_id': ObjectId('6a0439cb2373cf21fcd63688'),
  'Creation_date': '2026-05-13T08:43:55.690454',
  'HintText': "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.",
  'Category': 'General',
  'References_room': {'IdR': 2,
   'Name': 'sanctuary ',
   'IdD': 0,
   'Dungeon': 'Burghap, Prison of the Jealous Hippies'},
  'Publish_by': {'Email': 'adanherranz@example.org',
   'User_name': 'auroraespana',
   'CreationDate': '2022-01-02',
   'Country': 'es_ES'}},
 {'_id': ObjectId('6a0

#### `GET /room/{room_id}`

Este endpoint recibe el **id de una habitación** y devuelve la siguiente información:

- **`idR`**
- **`name`**
- **`inWP`**
- **`outWP`**

Además, incluye:

- El **número de monstruos de cada tipo** presentes en la habitación.
- El **total de oro** que valen los tesoros de la sala.
- Los **últimos 20 comentarios** realizados sobre esa habitación.

Cada comentario debe incluir:

- **`userName`**
- **`country`**
- **`creationDate`** del usuario que lo realizó
- **Texto**
- **Fecha de publicación**
- **Categoría** del comentario


*Antes:* Devolvía directamente la sala con su array `hints` embebido y extraía los últimos 20 comentarios.

*Ahora:*
- La consulta principal a `Rooms` ya no contiene los comentarios.
- Hay que realizar una **segunda consulta** a `Hints`, filtrando por `References_room.IdR == room_id`, ordenando por `Creation_date` descendente y limitando a 20.
- Si se desea evitar queries adicionales, se podría emplear un **`$lookup`** desde `Rooms` hacia `Hints`, pero dado que el endpoint solo requiere 20 comentarios, una query separada con un índice en `(References_room.IdR, Creation_date)` es muy eficiente.

In [58]:
hints = db["hints"]

In [59]:
def get_room(room_id):
    room = rooms.find_one({"room_id": room_id})
    last_comments = list(hints.find(
        {"References_room.IdR": room_id}
    ).sort("Creation_date", -1).limit(20))
    room["last_comments"] = last_comments
    return room

In [60]:
get_room(2)

{'_id': ObjectId('69df421577db15125cd10447'),
 'loot': None,
 'room_id': 2,
 'monsters': None,
 'room_name': 'sanctuary ',
 'dungeon_id': 0,
 'in_waypoint': None,
 'dungeon_name': 'Burghap, Prison of the Jealous Hippies',
 'out_waypoint': 'Perverted Volcano of Fomalhaut',
 'rooms_connected': [{'room_id': 3, 'room_name': 'game room '}],
 'last_comments': [{'_id': ObjectId('6a04468b2373cf21fcd6368c'),
   'Creation_date': '2026-05-13T09:38:19.667619',
   'HintText': "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.",
   'Category': 'General',
   'References_room': {'IdR': 2,
    'Name': 'sanctuary ',
    'IdD': 0,
    'Dungeon': 'Burghap, Prison of the Jealous Hippies'},
   'Publish_by': {'Email': 'adanherranz@example.org',
    'User_name': 'auroraespana',
    'CreationDate': '2022-01-02',
    'Country': 'es_ES'}},
  {'_id': ObjectId('6a043731034af1ad17aff839'),
   'Creation_date': '2022-07-11 15:28:56.000000',
   'Category': 'suggestion',
   'References_ro

#### `GET /dungeon/{dungeon_id}`

Este endpoint recibe el id de una mazmorra y devuelve información sobre una mazmorra del juego. 
Debe devolver: idM, name y lore. Además, este endpoint se utiliza para alimentar un grafo interactivo por lo que requiere la siguiente información: 

1) el nombre e id de cada habitación de la mazmorra; 
2) las conexiones entre habitaciones de la mazmorra; 
3) el id y el nombre de los monstruos que aparecen en cada habitación; 
4) el id y el nombre de los tesoros que aparecen en cada habitación; 
5) El número de comentarios de cada categoría que hay en cada habitación. 

*Antes:* Recorría las habitaciones de la mazmorra y, a partir de sus `hints` embebidos, contaba los comentarios por categoría.

*Ahora:* Existen dos enfoques:

1. **Sin precalcular (más caro):**  
   Se usa un pipeline de agregación que:
   - Filtra las salas de la mazmorra (`$match`).
   - Hace `$lookup` con `Hints` a través de `room_id`.
   - Desanida y agrupa por sala y categoría para contar.
   - Vuelve a agrupar para integrar los resultados en las salas.

2. **Con el patrón Computed:**  
   Si en el endpoint `POST /comment` se mantienen actualizados los contadores en el documento de la sala (campo `comments_by_category`), la obtención de la mazmorra no requiere ningún cambio: los contadores ya están desnormalizados en cada sala. La función simplemente sigue leyendo los documentos de las habitaciones y ya dispone de los números.

Dado que `GET /dungeon` tiene 1M de accesos diarios, la opción 2 es la única viable en producción. Por tanto, la función del endpoint apenas se modifica (salvo la eliminación de la lógica que contaba comentarios sobre arrays, que ya no existe).

In [61]:
def get_dungeon(dungeon_id):
    dungeon_rooms = list(rooms.find({"dungeon_id": dungeon_id}))
    for room in dungeon_rooms:
        room_id = room["room_id"]
        last_comments = list(hints.find(
            {"References_room.IdR": room_id}
        ).sort("Creation_date", -1).limit(20))
        room["last_comments"] = last_comments
    return dungeon_rooms

In [62]:
get_dungeon(1)

[{'_id': ObjectId('69df421577db15125cd10456'),
  'loot': None,
  'room_id': 33,
  'monsters': None,
  'room_name': 'drawing room of unknowns',
  'dungeon_id': 1,
  'in_waypoint': 'Jealous Volcano of Isengard',
  'dungeon_name': 'Burgstream, Culverts of the Bashful Sumo Wrestlers',
  'out_waypoint': None,
  'rooms_connected': [{'room_id': 45, 'room_name': 'great hall of rogues'}],
  'last_comments': [{'_id': ObjectId('6a043731034af1ad17aff928'),
    'Creation_date': '2021-06-02 17:50:08.000000',
    'Category': 'lore',
    'References_room': {'IdR': 33,
     'Name': 'drawing room of unknowns',
     'IdD': 1,
     'Dungeon': 'Burgstream, Culverts of the Bashful Sumo Wrestlers'},
    'Publish_by': {'Email': 'bolshakovapelageja@example.net',
     'User_name': 'pimenemeljanov',
     'CreationDate': '2021-01-09',
     'Country': 'ru_RU'}},
   {'_id': ObjectId('6a043731034af1ad17aff929'),
    'Creation_date': '2020-04-21 07:40:18.000000',
    'Category': 'lore',
    'References_room': {'IdR':

#### `GET /user/{email}`

Este endpoint recibe el email de un usuario y devuelve todos los campos de un usuario. Además, incluye 
los 20 últimos comentarios que ha realizado ese usuario. De cada comentario incluye, el texto, la fecha 
de creación, la categoría, el id (Room.IdR) y nombre (Room.name) de la habitación a la que hace 
referencia el comentario, el id (Dungeon.IdD) y nombre (Dungeon.name) de la mazmorra donde está la 
habitación. 

*Antes:* Devolvía los últimos 20 comentarios del usuario probablemente desde un array embebido en el documento del usuario.

*Ahora:*
- Se obtiene el usuario (sin array de comentarios).
- Se consulta la colección `Hints` filtrando por `Publish_by.Email`, ordenando por `Creation_date` descendente y limitando a 20.
- Se adjuntan esos comentarios al objeto de usuario devuelto.


In [63]:

def get_user(email):
    user = users.find_one({"email": email})
    last_comments = list(hints.find(
        {"Publish_by.Email": email}
    ).sort("Creation_date", -1).limit(20))
    user["last_comments"] = last_comments
    return user


In [ ]:
get_user("adanherranz@example.org")

{'_id': ObjectId('69df4218d280fa49533f7a7f'),
 'email': 'adanherranz@example.org',
 'country': 'es_ES',
 'user_name': 'auroraespana',
 'creation_date': '2022-01-02',
 'last_comments': [{'_id': ObjectId('6a04468b2373cf21fcd6368c'),
   'Creation_date': '2026-05-13T09:38:19.667619',
   'HintText': "Este es un comentario de ejemplo para la habitación 1 en la categoría 'General'.",
   'Category': 'General',
   'References_room': {'IdR': 2,
    'Name': 'sanctuary ',
    'IdD': 0,
    'Dungeon': 'Burghap, Prison of the Jealous Hippies'},
   'Publish_by': {'Email': 'adanherranz@example.org',
    'User_name': 'auroraespana',
    'CreationDate': '2022-01-02',
    'Country': 'es_ES'}}]}

### Impacto general del cambio

- **Ventajas:**  
  - Los comentarios quedan centralizados y normalizados, facilitando las consultas analíticas (como las del apartado 2) sin tener que recorrer arrays enormes.  
  - Se elimina la duplicación de datos (un mismo comentario ya no está en `Rooms` y en `Users`).  
  - Las operaciones de escritura (`POST`) son atómicas sobre un solo documento, evitando la actualización de arrays que pueden crecer indefinidamente.

- **Desventajas y contramedidas:**  
  - Las lecturas de comentarios ahora requieren consultas adicionales a `Hints`. Esto se mitiga con **índices compuestos** adecuados:  
    - `{References_room.IdR: 1, Creation_date: -1}` para `GET /room`.  
    - `{Publish_by.Email: 1, Creation_date: -1}` para `GET /user`.  
    - `{References_room.IdD: 1, Category: 1}` para agregaciones analíticas por mazmorra.  
  - El endpoint más crítico (`GET /dungeon`) no sufre penalización porque se apoya en los **contadores precalculados** (patrón Computed) que se mantienen durante la inserción de comentarios.
